# NB38 - trimmed trailing returns as a ranker: a vault-level screen

The track's luck diagnostics say the incumbent's result is carried by a few cycles and a few
names, and that vaults are admitted on trailing returns that may themselves be a few jumps.
This notebook asks the decision-time question directly, on vaults rather than portfolios: does
a trailing return computed with the vault's best k days REMOVED predict its forward 30-day
Sharpe better than the raw trailing return does? If it does not at the vault level, no ranker
built on it can beat the incumbent at the portfolio level, and the idea stops here.

**Focus is forward Sharpe**, not forward return: the operator wants steady profit, and a
trimmed score is expected to cost CAGR. Forward return, volatility and drawdown are reported
beside it. **Verdict: DIAGNOSTIC. Trimming does not make a better return ranker; it turns the
return leg into a volatility ranker, which the incumbent already has.**

**The panel is the archive, not the engine's candidate pool.** Every Hypercore vault with at
least the trailing window of history, a TVL of at least 7,500 USD and five price-changing marks
in the window is a candidate on every second day from 2026-04-01 (the polling-density break) to
the last date with a complete 30-day forward window: 17,215 candidate-dates, 367 vaults,
70 decisions to 2026-08-17, 79.9% of rows under 360 days old (cell 4). Stratwise
Multi-Asset Public (62 days old) and the other post-July vaults are too young for any
forward outcome and are NOT in the screen; they are shown in a current-snapshot comparison
(cell 10). No vault is selected, masked or tuned by name anywhere. Snapshot
`vault-prices.parquet` 255,548,076 bytes, sha256 `11e7c5e0103e1012`, last mark 2026-09-16 (cell 2).

**Based on:** [28-research-stability-signal-screen.ipynb](28-research-stability-signal-screen.ipynb)
and [34-research-calm-score-screen.ipynb](34-research-calm-score-screen.ipynb) for the
two-way cluster bootstrap and simultaneous bounds, re-implemented here without the engine.

## Method

Signals, read at T-1 over trailing windows of 45, 90 and 180 rows: annualised log return (raw
and with the best 3, 5 and 10 daily log returns removed), annualised Sharpe of daily log returns
(raw and trimmed the same way), Sortino, realised volatility. Direction 'high' for return and
Sharpe scores (higher is better), 'low' for volatility. Targets over (T, T + 30 d]: forward
Sharpe (primary), forward log return, forward volatility, forward max drawdown.

Inference: per date, Spearman across that date's candidates, signed so positive means "the
signal's good end had the better outcome", averaged over dates; one two-way cluster bootstrap
(15-decision circular date blocks x vault clusters, 500 draws, seed
20260916) shared across every hypothesis; studentised max-T simultaneous lower bounds
over the family of 30 signals on the primary target (critical value 2.59). The
decisive statistic is the PAIRED difference trimmed-minus-raw on the primary target, per
(window, k), on the same draws.

## Key new insights and what did we learn from this experiment?

**1. Nothing predicts a vault's next-30-day Sharpe well, and a raw trailing return does not
predict it at all.** The best of thirty signals is the raw 180-day Sharpe at rho
0.135 (unadjusted p 0.016); no signal clears the simultaneous
lower bound of zero over the family (best -0.032). Raw trailing return over 45 or 90
days is at -0.006 and -0.020 - nothing - and every signal's correlation
with forward RETURN is within 0.089 of zero (cell 6). A month
of a vault's Sharpe is mostly not in its past. The incumbent's ranker legs (45-day Sharpe,
360-day CAGR) are not in the top of this table either: `sharpe45_k0` sits at
0.048.

**2. Trimming the return leg helps - and the help is volatility, not return.** Removing the
best 10 of 90 days lifts the return signal from -0.020 to 0.086
on forward Sharpe; the paired difference is 0.106
[0.004, 0.209], p 0.044, and
at 45 days 0.101 [-0.010, 0.215] (cell 6).
But look at what the trimmed score correlates with: forward VOLATILITY at
0.630 (raw: 0.100) and forward drawdown at
0.567, exactly the profile of `vol90` itself (0.676 /
0.610, and 0.096 on forward Sharpe, the same as the trimmed
return). Removing a vault's best days removes most of what distinguishes a high-return vault
from a low-volatility one: the cross-sectional rank agreement between the raw and the k = 10
trimmed 90-day return at the current snapshot is 0.111 (cell 10) - a different
ordering, not a cleaned one. The trimmed return is a low-volatility ranker in disguise, and
NB28-NB37 already established what a volatility ranker does: it predicts forward volatility
(rho 0.7) and forward crashes, and it costs return when used to rank.

**3. Trimming the Sharpe leg does nothing.** Raw and trimmed trailing Sharpe rank the
cross-section almost identically (agreement 0.863 at k = 5, 0.947 at k = 3
on 180 days) and predict forward Sharpe identically: paired differences
-0.004 and -0.003 with intervals
straddling zero (cell 6). A Sharpe already divides by the jumps it is made of.

**4. The young cohort is where the return trim "works", for the same reason.** Among vaults
under 360 days (79.9% of rows), raw 90-day return is -0.039 on forward Sharpe and
trimmed k = 10 is 0.088, difference 0.127
[0.027, 0.237], p 0.020; among the
old (3,455 rows, 40 dates) every difference is inside its interval (cell 8). Young
vaults are where a few jumps most dominate a trailing return, so trimming re-orders them most -
towards low volatility.

**5. Stratwise, at the snapshot.** At the 45-day window on 2026-09-16, Stratwise's raw
annualised return is 30.5% (Sharpe 6.34, realised vol 4.8%); with its
best 5 days removed 6.9% and with 10 removed -1.9% - about 77% of its
45-day return is its best five days (cell 10). That is not unusual for the cohort, which is the point:
trimming demotes everyone, and on trimmed return Stratwise RISES from rank 122 to 18 of 220
because its peers are more concentrated still, while on Sharpe it sits at rank 13 raw and 17
trimmed. It cannot be screened for forward behaviour yet: 62 days of history give no
decision with both a trailing window and a complete forward window.

## Summary of results

Forward-Sharpe screen, all candidates (cell 6): signed Spearman, simultaneous lower bound over
30 signals, unadjusted add-one p.

| signal | rho fwd Sharpe | lower bound | p | rho fwd return | rho fwd vol (signed) |
|---|---|---|---|---|---|
| ret45 raw / k3 / k5 / k10 | -0.006 / 0.042 / 0.068 / 0.095 | -0.019 (k10) | 0.020 | 0.019 | 0.073 -> 0.694 |
| ret90 raw / k3 / k5 / k10 | -0.020 / 0.046 / 0.065 / 0.086 | -0.060 (k10) | 0.058 | 0.004 | 0.100 -> 0.630 |
| ret180 raw / k10 | 0.092 / 0.088 | -0.067 | 0.078 | 0.023 | 0.238 -> 0.566 |
| sharpe45 raw / k5 | 0.048 / 0.047 | -0.097 | 0.208 | -0.036 | 0.126 |
| sharpe90 raw / k5 | 0.031 / 0.028 | -0.106 | 0.273 | -0.066 | 0.122 |
| **sharpe180 raw / k5** | **0.135 / 0.132** | -0.032 | 0.016 | 0.034 | 0.228 |
| sortino 45 / 90 / 180 | 0.041 / 0.023 / 0.131 | -0.036 (180) | 0.020 | 0.032 | 0.252 |
| vol 45 / 90 / 180 (low is good) | 0.099 / 0.096 / 0.060 | -0.010 (45) | 0.012 | 0.042 | 0.730 |

Paired trimmed-minus-raw on forward Sharpe (cell 6; young cohort cell 8):

| | k = 3 | k = 5 | k = 10 |
|---|---|---|---|
| return, 45 d | 0.048 [-0.015, 0.112] | 0.074 [-0.012, 0.167] | 0.101 [-0.010, 0.215] |
| return, 90 d | 0.066 [0.000, 0.131] | 0.085 [-0.001, 0.167] | 0.106 [0.004, 0.209] |
| return, 180 d | 0.001 [-0.052, 0.058] | -0.002 [-0.070, 0.073] | -0.004 [-0.085, 0.084] |
| Sharpe, 90 d | -0.009 [-0.032, 0.017] | -0.004 [-0.042, 0.034] | 0.007 [-0.065, 0.077] |
| Sharpe, 180 d | -0.003 [-0.032, 0.025] | -0.003 [-0.045, 0.034] | -0.011 [-0.089, 0.050] |
| return, 90 d, young only | 0.079 [0.019, 0.144] | 0.101 [0.022, 0.189] | 0.127 [0.027, 0.237] |

**What this means for the ranker.** A trimmed-CAGR leg would not be a cleaner return leg; it
would be a second volatility leg beside the Sortino and the inverse-variance sizer, and NB32 and
NB37 have shown what ranking on stability does to return. The idea stops here, as the plan for
it said it should. The one signal with any persistence into next month's Sharpe is the 180-day
Sharpe itself - weak, and not separable from zero under a family-wise bound.

## Robustness of results

- The panel is built from the archive with a fixed rule (TVL, history, fresh marks) and no
  engine; it is therefore NOT the incumbent's candidate pool (no inclusion criteria, quarantine
  or momentum gate), and its per-date pools are larger (~246 candidates). The
  question asked is about vaults, so that is the right population; portfolio consequences are
  not claimed.
- Forward Sharpe is computed on 30 daily log returns of forward-filled marks; on post-break
  polling (15-17 marks a day) a zero-return day is a real flat day, not a gap, which is why the
  screen is restricted to decisions from 2026-04-01.
- Every hypothesis shares one bootstrap; paired differences are differences of the same draws,
  so their intervals are paired intervals. Simultaneous bounds are over the 30-signal family on
  the primary target only; the other targets are descriptive.
- 70 decisions of overlapping 30-day windows are about two independent months; the intervals
  say so. The young cohort's return-trim result (p 0.02) is the strongest single finding and it
  is explained by the volatility loading, not by cleaner return information.
- Stratwise's figures are a current snapshot at one window and are not evidence about its
  forward behaviour; the rule against selecting or tuning by name is unchanged.


## Part 0. Archive, provenance, constants


In [1]:
import hashlib, json, math
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import rankdata
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60); pd.set_option("display.max_rows", 200)

ARCHIVE = Path.home() / ".cache/tradingstrategy/vaults/downloads/vault-prices.parquet"
raw_bytes = ARCHIVE.read_bytes()
PROVENANCE = {"file": str(ARCHIVE), "bytes": len(raw_bytes), "sha256": hashlib.sha256(raw_bytes).hexdigest()}
del raw_bytes
HYPERCORE_CHAIN = 9999
STRATWISE = "0x0ff219ac20596b457558341bc410bc7a08a1394c"   # display only; never used to select or tune

WINDOWS = (45, 90, 180)
TRIMS = (0, 3, 5, 10)
FORWARD_DAYS = 30
POST_BREAK_START = pd.Timestamp("2026-04-01")
DECISION_STEP_DAYS = 2
MIN_TVL_USD = 7_500.0
MIN_FRESH = 5
MIN_CANDIDATES = 8
MIN_DATES = 40
YOUNG_DAYS = 360          # the incumbent's CAGR leg cannot score a vault younger than this
DRAWS = 500
DATE_BLOCK = 15
SEED = 20260916
LEVEL = 0.95

df = pd.read_parquet(ARCHIVE, columns=["address", "chain", "share_price", "total_assets", "name"])
df = df[df["chain"] == HYPERCORE_CHAIN].reset_index()
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df[df["timestamp"] >= pd.Timestamp("2025-06-01")].sort_values(["address", "timestamp"])
df["date"] = df["timestamp"].dt.floor("D")
daily = df.groupby(["address", "date"])[["share_price", "total_assets"]].last().reset_index()
names = df.groupby("address")["name"].last()
LAST_MARK = df["timestamp"].max()
display(pd.Series({**PROVENANCE, "last_mark": str(LAST_MARK), "hypercore_vaults": daily["address"].nunique(),
                   "daily_rows": len(daily)}, name="value").to_frame())


,value
file,/Users/moo/.cache/tradingstrategy/vaults/downl...
bytes,255548076
sha256,11e7c5e0103e10125a27fcb2ed58febd394cab3e317771...
last_mark,2026-09-16 07:25:06.705000
hypercore_vaults,604
daily_rows,111006


## Part 1. The panel

One row per (decision date, vault). Trailing series are forward-filled daily marks (a day
without a mark repeats the last one, a zero log return); trimming removes the k LARGEST daily
log returns inside the window before annualising, so a vault whose window return is three jumps
loses most of it. Forward series are the same daily grid over the next 30 days.


In [2]:
def trailing_scores(r: np.ndarray) -> dict:
    """Raw and trimmed annualised log return and Sharpe, Sortino and volatility of one window."""
    out = {}
    n = len(r)
    order = np.sort(r)
    for k in TRIMS:
        kept = order[:n - k] if k else order
        ann = 365.0 / n
        out[f"ret_k{k}"] = float(kept.sum() * ann)
        sd = float(kept.std(ddof=1)) if len(kept) > 2 else float("nan")
        out[f"sharpe_k{k}"] = float(kept.mean() / sd * math.sqrt(365.0)) if sd and sd > 0 else float("nan")
    downside = np.sqrt(np.mean(np.clip(r, None, 0.0) ** 2))
    out["sortino"] = float(r.mean() / downside * math.sqrt(365.0)) if downside > 0 else float("nan")
    out["vol"] = float(r.std(ddof=1) * math.sqrt(365.0))
    return out


def forward_targets(r: np.ndarray, prices: np.ndarray) -> dict:
    sd = float(r.std(ddof=1)) if len(r) > 2 else float("nan")
    path = np.concatenate([[0.0], np.cumsum(r)])
    return {"fwd_return": float(r.sum()),
            "fwd_sharpe": float(r.mean() / sd * math.sqrt(365.0)) if sd and sd > 0 else float("nan"),
            "fwd_vol": float(sd * math.sqrt(365.0)) if sd == sd else float("nan"),
            "fwd_max_dd": float(np.min(path - np.maximum.accumulate(path)))}


last_decision = (LAST_MARK.floor("D") - pd.Timedelta(days=FORWARD_DAYS))
decisions = pd.date_range(POST_BREAK_START, last_decision, freq=f"{DECISION_STEP_DAYS}D")
rows = []
for address, g in daily.groupby("address"):
    g = g.set_index("date")
    grid = pd.date_range(g.index.min(), LAST_MARK.floor("D"), freq="D")
    full = g.reindex(grid).ffill()
    p = full["share_price"].where(full["share_price"] > 0)
    tvl = full["total_assets"]
    r = np.log(p).diff()
    born = g.index.min()
    for t in decisions:
        if t not in full.index:
            continue
        t1 = t - pd.Timedelta(days=1)
        if not (tvl.get(t1, 0.0) >= MIN_TVL_USD):
            continue
        age = int((t - born).days)
        row = {"address": address, "date": t, "age_days": age, "young": age < YOUNG_DAYS}
        scored_any = False
        for w in WINDOWS:
            win = r[(r.index > t1 - pd.Timedelta(days=w)) & (r.index <= t1)].dropna().to_numpy()
            if len(win) < w or int((np.abs(win) > 0).sum()) < MIN_FRESH:
                for k in TRIMS:
                    row[f"ret{w}_k{k}"] = np.nan; row[f"sharpe{w}_k{k}"] = np.nan
                row[f"sortino{w}"] = np.nan; row[f"vol{w}"] = np.nan
                row[f"fresh{w}"] = int((np.abs(win) > 0).sum()) if len(win) else 0
                continue
            scored_any = True
            s = trailing_scores(win)
            for key, value in s.items():
                row[f"{key.split('_')[0]}{w}_{key.split('_')[1]}" if "_" in key else f"{key}{w}"] = value
            row[f"fresh{w}"] = int((np.abs(win) > 0).sum())
        if not scored_any:
            continue
        fwd = r[(r.index > t) & (r.index <= t + pd.Timedelta(days=FORWARD_DAYS))].dropna().to_numpy()
        if len(fwd) < FORWARD_DAYS:
            continue
        row.update(forward_targets(fwd, None))
        rows.append(row)
panel = pd.DataFrame(rows)
panel["name"] = panel["address"].map(names)
print(f"panel: {len(panel):,} rows, {panel['address'].nunique()} vaults, {panel['date'].nunique()} decisions "
      f"{panel['date'].min().date()} to {panel['date'].max().date()}; young (< {YOUNG_DAYS} d) share of rows "
      f"{panel['young'].mean():.1%}")
SIGNALS = []
for w in WINDOWS:
    for k in TRIMS:
        SIGNALS.append({"name": f"ret{w}_k{k}", "direction": "high", "window": w, "k": k, "family": "return"})
    for k in TRIMS:
        SIGNALS.append({"name": f"sharpe{w}_k{k}", "direction": "high", "window": w, "k": k, "family": "sharpe"})
    SIGNALS.append({"name": f"sortino{w}", "direction": "high", "window": w, "k": None, "family": "sortino"})
    SIGNALS.append({"name": f"vol{w}", "direction": "low", "window": w, "k": None, "family": "vol"})
SIGNAL_NAMES = [s["name"] for s in SIGNALS]
SIGNAL_SIGN = {s["name"]: (1.0 if s["direction"] == "high" else -1.0) for s in SIGNALS}
TARGETS = ["fwd_sharpe", "fwd_return", "fwd_vol", "fwd_max_dd"]
#: positive = the signal's good end had the better outcome: higher Sharpe/return, LOWER vol, shallower (larger) max DD
TARGET_SIGN = {"fwd_sharpe": 1.0, "fwd_return": 1.0, "fwd_vol": -1.0, "fwd_max_dd": 1.0}
coverage = pd.DataFrame({s: np.isfinite(panel[s]).mean() for s in SIGNAL_NAMES}, index=["finite_share"]).T
display(coverage.round(3).T)
display(panel[TARGETS].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(4))


panel: 17,215 rows, 367 vaults, 70 decisions 2026-04-01 to 2026-08-17; young (< 360 d) share of rows 79.9%


,ret45_k0,ret45_k3,ret45_k5,ret45_k10,sharpe45_k0,sharpe45_k3,sharpe45_k5,sharpe45_k10,sortino45,vol45,ret90_k0,ret90_k3,ret90_k5,ret90_k10,sharpe90_k0,sharpe90_k3,sharpe90_k5,sharpe90_k10,sortino90,vol90,ret180_k0,ret180_k3,ret180_k5,ret180_k10,sharpe180_k0,sharpe180_k3,sharpe180_k5,sharpe180_k10,sortino180,vol180
finite_share,0.941,0.941,0.941,0.941,0.941,0.941,0.941,0.94,0.937,0.941,0.86,0.86,0.86,0.86,0.86,0.86,0.86,0.86,0.858,0.86,0.645,0.645,0.645,0.645,0.645,0.645,0.645,0.645,0.645,0.645


,fwd_sharpe,fwd_return,fwd_vol,fwd_max_dd
count,16476.0000,17215.0000,17215.0000,17215.0000
mean,0.6454,-0.0651,0.6462,-0.2139
std,9.3954,0.4793,1.0363,0.4873
min,-14.2720,-13.1463,0.0000,-13.1463
5%,-6.2436,-0.6364,0.0000,-0.9088
25%,-3.0606,-0.0577,0.0981,-0.2033
50%,0.0833,0.0000,0.2917,-0.0689
75%,3.1583,0.0554,0.7946,-0.0164
95%,6.8098,0.3159,2.4010,0.0000
max,234.4167,1.9923,16.4087,0.0000


## Part 2. One shared bootstrap, every hypothesis

Per date and signal, the complete-case Spearman against each target; equal-weight mean over
dates. The bootstrap resamples 15-decision circular date blocks and vault clusters together, and
every hypothesis - every signal, every target, and every trimmed-minus-raw difference - is a
function of the same draws.


In [3]:
def per_date_blocks(frame: pd.DataFrame) -> dict:
    """{date: {'vault': array, 'values': (n, S+T) float array}} with NaN where a signal is missing."""
    out = {}
    cols = SIGNAL_NAMES + TARGETS
    for date, g in frame.groupby("date"):
        out[pd.Timestamp(date)] = {"vault": g["address"].to_numpy(), "values": g[cols].to_numpy(dtype=float)}
    return out


def date_statistics(values: np.ndarray) -> np.ndarray:
    """(S, T) signed Spearman on the complete cases of each (signal, target) pair; NaN if fewer than
    MIN_CANDIDATES rows or a constant column."""
    S, T = len(SIGNAL_NAMES), len(TARGETS)
    out = np.full((S, T), np.nan)
    targets = values[:, S:]
    for i, name in enumerate(SIGNAL_NAMES):
        x = values[:, i]
        for j, target in enumerate(TARGETS):
            y = targets[:, j]
            ok = np.isfinite(x) & np.isfinite(y)
            if ok.sum() < MIN_CANDIDATES:
                continue
            xr, yr = rankdata(x[ok]), rankdata(y[ok])
            if np.ptp(xr) == 0 or np.ptp(yr) == 0:
                continue
            xc, yc = xr - xr.mean(), yr - yr.mean()
            rho = float((xc * yc).sum() / math.sqrt((xc ** 2).sum() * (yc ** 2).sum()))
            out[i, j] = SIGNAL_SIGN[name] * TARGET_SIGN[target] * rho
    return out


def mean_over_dates(blocks: dict, dates: list, counts: dict | None = None) -> tuple:
    S, T = len(SIGNAL_NAMES), len(TARGETS)
    total, n = np.zeros((S, T)), np.zeros((S, T))
    for date in dates:
        block = blocks.get(date)
        if block is None:
            continue
        values = block["values"]
        if counts is not None:
            repeats = np.array([counts.get(v, 0) for v in block["vault"]], dtype=int)
            if repeats.sum() < MIN_CANDIDATES:
                continue
            values = np.repeat(values, repeats, axis=0)
        stats = date_statistics(values)
        finite = np.isfinite(stats)
        total[finite] += stats[finite]
        n[finite] += 1
    with np.errstate(invalid="ignore"):
        return np.where(n > 0, total / np.maximum(n, 1), np.nan), n


def bootstrap(frame: pd.DataFrame, draws: int = DRAWS, seed: int = SEED, verbose: bool = True) -> dict:
    blocks = per_date_blocks(frame)
    dates = sorted(blocks)
    vaults = sorted(frame["address"].unique())
    observed, n_dates = mean_over_dates(blocks, dates)
    rng = np.random.default_rng(seed)
    n_blocks = int(math.ceil(len(dates) / DATE_BLOCK))
    reps = np.full((draws,) + observed.shape, np.nan)
    for d in range(draws):
        starts = rng.integers(0, len(dates), size=n_blocks)
        index = np.concatenate([(np.arange(s, s + DATE_BLOCK) % len(dates)) for s in starts])[:len(dates)]
        drawn_vaults = rng.choice(len(vaults), size=len(vaults), replace=True)
        counts = {}
        for v in drawn_vaults:
            counts[vaults[v]] = counts.get(vaults[v], 0) + 1
        reps[d], _ = mean_over_dates(blocks, [dates[i] for i in index], counts)
        if verbose and (d + 1) % 100 == 0:
            print(f"  draw {d + 1}/{draws}")
    return {"observed": observed, "draws": reps, "n_dates": n_dates, "dates": dates, "rows": len(frame)}


def simultaneous_lower(observed: np.ndarray, reps: np.ndarray, level: float = LEVEL) -> dict:
    """One-sided simultaneous lower bounds by studentised max-T over the finite family, on
    complete-family draws only; add-one p for theta <= 0."""
    se = np.nanstd(reps, axis=0, ddof=1)
    member = np.isfinite(observed) & np.isfinite(se) & (se > 0)
    stud = (reps - observed[None, :]) / np.where(se > 0, se, np.nan)[None, :]
    complete = np.isfinite(stud[:, member]).all(axis=1) if member.any() else np.zeros(len(reps), dtype=bool)
    per_draw_max = stud[complete][:, member].max(axis=1) if complete.any() else np.array([])
    critical = float(np.percentile(per_draw_max, level * 100.0)) if len(per_draw_max) >= 100 else float("nan")
    lower = np.where(member, observed - critical * se, np.nan)
    unadjusted = np.nanpercentile(reps, (1 - level) * 100.0, axis=0)
    centred = reps - observed[None, :]
    p = (1.0 + (centred >= observed[None, :]).sum(axis=0)) / (len(reps) + 1.0)
    return {"se": se, "critical": critical, "lower": lower, "lower_unadjusted": unadjusted, "p": p,
            "family_size": int(member.sum()), "complete_draws": int(len(per_draw_max))}


def screen(frame: pd.DataFrame, label: str, verbose: bool = True) -> dict:
    print(f"{label}: {len(frame):,} rows, {frame['date'].nunique()} decisions, {frame['address'].nunique()} vaults")
    boot = bootstrap(frame, verbose=verbose)
    S, T = len(SIGNAL_NAMES), len(TARGETS)
    j = TARGETS.index("fwd_sharpe")
    fam = simultaneous_lower(boot["observed"][:, j], boot["draws"][:, :, j])
    rows = []
    for i, s in enumerate(SIGNALS):
        row = {"signal": s["name"], "family": s["family"], "window": s["window"], "k": s["k"],
               "dates": int(boot["n_dates"][i, j]), "evaluated": bool(boot["n_dates"][i, j] >= MIN_DATES and fam["se"][i] > 0)}
        for t_idx, target in enumerate(TARGETS):
            row[f"rho_{target}"] = boot["observed"][i, t_idx]
        row["se_sharpe"] = fam["se"][i]
        row["lo_sharpe_simultaneous"] = fam["lower"][i]
        row["lo_sharpe_unadjusted"] = fam["lower_unadjusted"][i]
        row["p_sharpe"] = fam["p"][i]
        rows.append(row)
    table = pd.DataFrame(rows).set_index("signal")
    # Paired trimmed-minus-raw differences on the primary target, on the same draws.
    diffs = []
    for w in WINDOWS:
        for fam_name in ("ret", "sharpe"):
            base = SIGNAL_NAMES.index(f"{fam_name}{w}_k0")
            for k in TRIMS[1:]:
                idx = SIGNAL_NAMES.index(f"{fam_name}{w}_k{k}")
                obs = boot["observed"][idx, j] - boot["observed"][base, j]
                rep = boot["draws"][:, idx, j] - boot["draws"][:, base, j]
                rep = rep[np.isfinite(rep)]
                diffs.append({"family": fam_name, "window": w, "k": k, "trimmed_rho": boot["observed"][idx, j],
                              "raw_rho": boot["observed"][base, j], "difference": obs,
                              "ci_lo": float(np.percentile(rep, 2.5)) if len(rep) >= 100 else np.nan,
                              "ci_hi": float(np.percentile(rep, 97.5)) if len(rep) >= 100 else np.nan,
                              "p_two_sided": float(2 * min((rep - obs >= obs).mean(), (rep - obs <= obs).mean())) if len(rep) >= 100 else np.nan,
                              "draws": int(len(rep))})
    paired = pd.DataFrame(diffs)
    return {"label": label, "table": table, "paired": paired, "critical": fam["critical"],
            "family_size": fam["family_size"], "complete_draws": fam["complete_draws"], "boot": boot}


full = screen(panel, "all candidates")
print(f"\nprimary family: {full['family_size']} signals, critical {full['critical']:.4f} on {full['complete_draws']} complete draws")
display(full["table"][["family", "window", "k", "dates", "evaluated", "rho_fwd_sharpe", "se_sharpe", "lo_sharpe_simultaneous",
                       "lo_sharpe_unadjusted", "p_sharpe", "rho_fwd_return", "rho_fwd_vol", "rho_fwd_max_dd"]].round(4))
print("\nPAIRED trimmed - raw on forward Sharpe (95% percentile interval on shared draws):")
display(full["paired"].round(4))


all candidates: 17,215 rows, 70 decisions, 367 vaults


  draw 100/500


  draw 200/500


  draw 300/500


  draw 400/500


  draw 500/500

primary family: 30 signals, critical 2.5894 on 500 complete draws


,family,window,k,dates,evaluated,rho_fwd_sharpe,se_sharpe,lo_sharpe_simultaneous,lo_sharpe_unadjusted,p_sharpe,rho_fwd_return,rho_fwd_vol,rho_fwd_max_dd
signal,,,,,,,,,,,,,
ret45_k0,return,45,0.0,70,True,-0.0061,0.0528,-0.1429,-0.0930,0.5808,-0.0514,0.0729,0.0464
ret45_k3,return,45,3.0,70,True,0.0415,0.0461,-0.0779,-0.0386,0.1796,-0.0346,0.4794,0.4174
ret45_k5,return,45,5.0,70,True,0.0675,0.0445,-0.0478,-0.0083,0.0599,-0.0116,0.5946,0.5298
ret45_k10,return,45,10.0,70,True,0.0947,0.0439,-0.0189,0.0196,0.0200,0.0192,0.6937,0.6305
sharpe45_k0,sharpe,45,0.0,70,True,0.0481,0.0559,-0.0967,-0.0378,0.2076,-0.0364,0.1258,0.1017
sharpe45_k3,sharpe,45,3.0,70,True,0.0427,0.0576,-0.1063,-0.0466,0.2435,-0.0401,0.0997,0.0761
sharpe45_k5,sharpe,45,5.0,70,True,0.0467,0.0558,-0.0978,-0.0442,0.2136,-0.0429,0.1287,0.1068
sharpe45_k10,sharpe,45,10.0,70,True,0.0488,0.0452,-0.0681,-0.0264,0.1417,-0.0380,0.2256,0.2138
sortino45,sortino,45,NaN,70,True,0.0414,0.0557,-0.1027,-0.0480,0.2156,-0.0336,0.1327,0.1081



PAIRED trimmed - raw on forward Sharpe (95% percentile interval on shared draws):


,family,window,k,trimmed_rho,raw_rho,difference,ci_lo,ci_hi,p_two_sided,draws
0,ret,45,3,0.0415,-0.0061,0.0476,-0.0152,0.1119,0.132,500
1,ret,45,5,0.0675,-0.0061,0.0737,-0.0117,0.1665,0.100,500
2,ret,45,10,0.0947,-0.0061,0.1009,-0.0096,0.2147,0.076,500
3,sharpe,45,3,0.0427,0.0481,-0.0054,-0.0295,0.0183,0.640,500
4,sharpe,45,5,0.0467,0.0481,-0.0014,-0.0437,0.0358,0.936,500
5,sharpe,45,10,0.0488,0.0481,0.0007,-0.0737,0.0808,0.964,500
6,ret,90,3,0.0463,-0.0202,0.0665,0.0003,0.1313,0.044,500
7,ret,90,5,0.0645,-0.0202,0.0847,-0.0011,0.1670,0.044,500
8,ret,90,10,0.0863,-0.0202,0.1064,0.0041,0.2085,0.044,500
9,sharpe,90,3,0.0227,0.0314,-0.0088,-0.0322,0.0168,0.476,500


## Part 3. The young cohort and the old

Vaults under 360 days old are unscorable by the incumbent's CAGR leg; they are where a
shorter-window ranker would matter most. The screen is repeated on the young rows and on the
rest, separately, so the two are not averaged into each other.


In [4]:
young = screen(panel[panel["young"]], "young (< 360 days)", verbose=False)
old = screen(panel[~panel["young"]], "old (>= 360 days)", verbose=False)
COLS = ["window", "k", "dates", "evaluated", "rho_fwd_sharpe", "lo_sharpe_simultaneous", "p_sharpe", "rho_fwd_return", "rho_fwd_vol"]
for res in (young, old):
    print(f"\n{res['label']}: family {res['family_size']}, critical {res['critical']:.4f}, complete draws {res['complete_draws']}")
    display(res["table"][COLS].round(4))
    print("paired trimmed - raw on forward Sharpe:")
    display(res["paired"].round(4))


young (< 360 days): 13,760 rows, 70 decisions, 360 vaults


old (>= 360 days): 3,455 rows, 40 decisions, 114 vaults



young (< 360 days): family 30, critical 2.5849, complete draws 500


,window,k,dates,evaluated,rho_fwd_sharpe,lo_sharpe_simultaneous,p_sharpe,rho_fwd_return,rho_fwd_vol
signal,,,,,,,,,
ret45_k0,45,0.0,70,True,-0.0196,-0.1752,0.6148,-0.0577,0.0616
ret45_k3,45,3.0,70,True,0.0370,-0.1026,0.2535,-0.0250,0.4666
ret45_k5,45,5.0,70,True,0.0676,-0.0660,0.0798,0.0035,0.5853
ret45_k10,45,10.0,70,True,0.0997,-0.0318,0.0180,0.0380,0.6856
sharpe45_k0,45,0.0,70,True,0.0405,-0.1260,0.2655,-0.0385,0.1174
sharpe45_k3,45,3.0,70,True,0.0359,-0.1333,0.2934,-0.0430,0.0921
sharpe45_k5,45,5.0,70,True,0.0405,-0.1251,0.2675,-0.0467,0.1218
sharpe45_k10,45,10.0,70,True,0.0403,-0.1026,0.2475,-0.0453,0.2211
sortino45,45,NaN,70,True,0.0289,-0.1346,0.3234,-0.0366,0.1219


paired trimmed - raw on forward Sharpe:


,family,window,k,trimmed_rho,raw_rho,difference,ci_lo,ci_hi,p_two_sided,draws
0,ret,45,3,0.0370,-0.0196,0.0566,-0.0069,0.1269,0.148,500
1,ret,45,5,0.0676,-0.0196,0.0873,-0.0016,0.1830,0.092,500
2,ret,45,10,0.0997,-0.0196,0.1193,0.0081,0.2387,0.052,500
3,sharpe,45,3,0.0359,0.0405,-0.0046,-0.0352,0.0253,0.752,500
4,sharpe,45,5,0.0405,0.0405,0.0001,-0.0431,0.0456,0.968,500
5,sharpe,45,10,0.0403,0.0405,-0.0001,-0.0747,0.0833,0.956,500
6,ret,90,3,0.0398,-0.0391,0.0789,0.0192,0.1439,0.016,500
7,ret,90,5,0.0621,-0.0391,0.1011,0.0222,0.1895,0.020,500
8,ret,90,10,0.0882,-0.0391,0.1272,0.0268,0.2365,0.020,500
9,sharpe,90,3,0.0150,0.0212,-0.0063,-0.0374,0.0287,0.696,500



old (>= 360 days): family 30, critical 2.7077, complete draws 500


,window,k,dates,evaluated,rho_fwd_sharpe,lo_sharpe_simultaneous,p_sharpe,rho_fwd_return,rho_fwd_vol
signal,,,,,,,,,
ret45_k0,45,0.0,40,True,0.0196,-0.1594,0.3713,-0.0419,0.1754
ret45_k3,45,3.0,40,True,0.0275,-0.1765,0.3613,-0.0920,0.5594
ret45_k5,45,5.0,40,True,0.0414,-0.1623,0.3114,-0.0860,0.6534
ret45_k10,45,10.0,40,True,0.0517,-0.1559,0.2495,-0.0698,0.7526
sharpe45_k0,45,0.0,40,True,0.0421,-0.1712,0.2754,-0.0545,0.2004
sharpe45_k3,45,3.0,40,True,0.0462,-0.1769,0.2794,-0.0449,0.1621
sharpe45_k5,45,5.0,40,True,0.0609,-0.1522,0.1976,-0.0319,0.1798
sharpe45_k10,45,10.0,40,True,0.0825,-0.1052,0.1058,-0.0037,0.2369
sortino45,45,NaN,40,True,0.0428,-0.1720,0.2834,-0.0532,0.2137


paired trimmed - raw on forward Sharpe:


,family,window,k,trimmed_rho,raw_rho,difference,ci_lo,ci_hi,p_two_sided,draws
0,ret,45,3,0.0275,0.0196,0.0079,-0.1003,0.1195,0.880,500
1,ret,45,5,0.0414,0.0196,0.0217,-0.1047,0.1574,0.728,500
2,ret,45,10,0.0517,0.0196,0.0320,-0.1160,0.1875,0.676,500
3,sharpe,45,3,0.0462,0.0421,0.0041,-0.0424,0.0430,0.824,500
4,sharpe,45,5,0.0609,0.0421,0.0188,-0.0387,0.0709,0.424,500
5,sharpe,45,10,0.0825,0.0421,0.0405,-0.0603,0.1589,0.416,500
6,ret,90,3,0.0133,-0.0112,0.0245,-0.1003,0.1881,0.736,500
7,ret,90,5,0.0235,-0.0112,0.0347,-0.1182,0.2157,0.712,500
8,ret,90,10,0.0378,-0.0112,0.0490,-0.1338,0.2477,0.640,500
9,sharpe,90,3,0.0209,0.0239,-0.0030,-0.0564,0.0396,0.808,500


## Part 4. Stratwise and the post-July cohort: a current-snapshot comparison

Stratwise Multi-Asset Public has 63 days of history at this archive. It cannot appear in the
screen: no decision date gives it a 45-day trailing window AND a complete 30-day forward window.
What can be shown is how raw and trimmed trailing scores rank it and the other young vaults on
the LATEST date at which each window can be computed, against every vault scorable on that
date. This is description; nothing here is evidence about forward behaviour.


In [5]:
snapshot_rows = []
snap_dates = {}
for w in WINDOWS:
    t = LAST_MARK.floor("D")
    snap_dates[w] = t
    for address, g in daily.groupby("address"):
        g = g.set_index("date")
        grid = pd.date_range(g.index.min(), t, freq="D")
        full_ = g.reindex(grid).ffill()
        p = full_["share_price"].where(full_["share_price"] > 0)
        if not (full_["total_assets"].iloc[-1] >= MIN_TVL_USD):
            continue
        r = np.log(p).diff()
        win = r[(r.index > t - pd.Timedelta(days=w)) & (r.index <= t)].dropna().to_numpy()
        if len(win) < w or int((np.abs(win) > 0).sum()) < MIN_FRESH:
            continue
        s = trailing_scores(win)
        snapshot_rows.append({"window": w, "address": address, "name": names.get(address, ""),
                              "age_days": int((t - g.index.min()).days), "tvl": float(full_["total_assets"].iloc[-1]),
                              "fresh": int((np.abs(win) > 0).sum()), **s})
snapshot = pd.DataFrame(snapshot_rows)
for w in WINDOWS:
    sub = snapshot[snapshot["window"] == w].copy()
    for col in ["ret_k0", "ret_k5", "sharpe_k0", "sharpe_k5"]:
        sub[f"rank_{col}"] = sub[col].rank(ascending=False, method="min")
    sub = sub.sort_values("sharpe_k5", ascending=False)
    print(f"\nwindow {w} rows at {snap_dates[w].date()}: {len(sub)} scorable vaults; post-July (age < 75 d): {(sub['age_days'] < 75).sum()}")
    show = sub[(sub["age_days"] < 120) | (sub["address"] == STRATWISE)].head(25)
    display(show[["name", "age_days", "tvl", "fresh", "ret_k0", "ret_k5", "sharpe_k0", "sharpe_k5", "sortino", "vol",
                  "rank_ret_k0", "rank_ret_k5", "rank_sharpe_k0", "rank_sharpe_k5"]].round(3).set_index("name"))
    if (sub["address"] == STRATWISE).any():
        sw = sub[sub["address"] == STRATWISE].iloc[0]
        print(f"  Stratwise at window {w}: raw return {sw['ret_k0']:+.3f} (rank {int(sw['rank_ret_k0'])}/{len(sub)}), trimmed k=5 "
              f"{sw['ret_k5']:+.3f} (rank {int(sw['rank_ret_k5'])}); raw Sharpe {sw['sharpe_k0']:.2f} (rank {int(sw['rank_sharpe_k0'])}), "
              f"trimmed k=5 {sw['sharpe_k5']:.2f} (rank {int(sw['rank_sharpe_k5'])})")
    else:
        print(f"  Stratwise has no {w}-row window at {snap_dates[w].date()}")
# How much trimming moves the cross-sectional ordering at all, per window: rank correlation raw vs trimmed.
agreement = []
for w in WINDOWS:
    sub = snapshot[snapshot["window"] == w]
    for fam_name in ("ret", "sharpe"):
        for k in TRIMS[1:]:
            agreement.append({"window": w, "family": fam_name, "k": k,
                              "spearman_raw_vs_trimmed": float(sub[f"{fam_name}_k0"].corr(sub[f"{fam_name}_k{k}"], method="spearman")),
                              "vaults": len(sub)})
display(pd.DataFrame(agreement).round(3))



window 45 rows at 2026-09-16: 220 scorable vaults; post-July (age < 75 d): 2


,age_days,tvl,fresh,ret_k0,ret_k5,sharpe_k0,sharpe_k5,sortino,vol,rank_ret_k0,rank_ret_k5,rank_sharpe_k0,rank_sharpe_k5
name,,,,,,,,,,,,,
[Bee] Line,55,28144.567,44,2.423,1.127,8.339,5.160,16.773,0.291,46.0,3.0,7.0,8.0
Stratwise Multi-Asset Public,62,186036.795,45,0.305,0.069,6.339,2.260,11.263,0.048,122.0,18.0,13.0,17.0
DOUBLETOP Vault,89,5182695.968,45,1.098,-0.024,4.217,-0.188,11.008,0.260,74.0,36.0,38.0,29.0
Kairos Fi,111,43046.652,37,2.253,-0.993,3.313,-2.370,6.103,0.680,48.0,123.0,59.0,62.0
korea wins again,83,112252.183,45,-1.987,-4.191,-2.260,-5.763,-2.665,0.879,204.0,188.0,186.0,156.0
NEET WORLD ORDER,113,340186.184,45,-1.518,-6.359,-1.173,-7.369,-1.603,1.294,198.0,206.0,177.0,200.0
Nova Quant,99,29218.589,45,-0.742,-1.063,-5.246,-9.099,-5.540,0.141,188.0,125.0,215.0,214.0


  Stratwise at window 45: raw return +0.305 (rank 122/220), trimmed k=5 +0.069 (rank 18); raw Sharpe 6.34 (rank 13), trimmed k=5 2.26 (rank 17)

window 90 rows at 2026-09-16: 227 scorable vaults; post-July (age < 75 d): 0


,age_days,tvl,fresh,ret_k0,ret_k5,sharpe_k0,sharpe_k5,sortino,vol,rank_ret_k0,rank_ret_k5,rank_sharpe_k0,rank_sharpe_k5
name,,,,,,,,,,,,,
Kairos Fi,111,43046.652,44,0.975,-0.648,2.000,-2.083,3.653,0.488,38.0,118.0,46.0,82.0
NEET WORLD ORDER,113,340186.184,74,-0.881,-3.361,-0.838,-4.103,-1.090,1.051,200.0,196.0,173.0,168.0
Nova Quant,99,29218.589,90,-0.154,-0.690,-0.719,-4.288,-1.013,0.213,168.0,122.0,167.0,178.0


  Stratwise has no 90-row window at 2026-09-16

window 180 rows at 2026-09-16: 198 scorable vaults; post-July (age < 75 d): 0


,age_days,tvl,fresh,ret_k0,ret_k5,sharpe_k0,sharpe_k5,sortino,vol,rank_ret_k0,rank_ret_k5,rank_sharpe_k0,rank_sharpe_k5
name,,,,,,,,,,,,,


  Stratwise has no 180-row window at 2026-09-16


,window,family,k,spearman_raw_vs_trimmed,vaults
0,45,ret,3,0.498,220
1,45,ret,5,0.230,220
2,45,ret,10,-0.027,220
3,45,sharpe,3,0.942,220
4,45,sharpe,5,0.874,220
5,45,sharpe,10,0.596,220
6,90,ret,3,0.512,227
7,90,ret,5,0.311,227
8,90,ret,10,0.111,227
9,90,sharpe,3,0.908,227


## Part 5. Manifest


In [6]:
def table_records(res):
    return {"table": res["table"].round(6).to_dict(orient="index"), "paired": res["paired"].round(6).to_dict(orient="records"),
            "critical": res["critical"], "family_size": res["family_size"], "complete_draws": res["complete_draws"],
            "rows": int(res["boot"]["rows"]), "decisions": int(len(res["boot"]["dates"]))}

manifest = {
    "verdict": "DIAGNOSTIC - a vault-level screen, not a result",
    "provenance": {**PROVENANCE, "last_mark": str(LAST_MARK)},
    "constants": {"windows": list(WINDOWS), "trims": list(TRIMS), "forward_days": FORWARD_DAYS, "post_break_start": str(POST_BREAK_START.date()),
                  "min_tvl_usd": MIN_TVL_USD, "min_fresh": MIN_FRESH, "min_candidates": MIN_CANDIDATES, "min_dates": MIN_DATES,
                  "young_days": YOUNG_DAYS, "draws": DRAWS, "date_block": DATE_BLOCK, "seed": SEED},
    "panel": {"rows": int(len(panel)), "vaults": int(panel["address"].nunique()), "decisions": int(panel["date"].nunique()),
              "first": str(panel["date"].min().date()), "last": str(panel["date"].max().date()), "young_share": float(panel["young"].mean())},
    "coverage": coverage["finite_share"].round(6).to_dict(),
    "screens": {"all": table_records(full), "young": table_records(young), "old": table_records(old)},
    "snapshot_dates": {str(w): str(t.date()) for w, t in snap_dates.items()},
    "snapshot_agreement": pd.DataFrame(agreement).round(6).to_dict(orient="records"),
    "stratwise": {str(w): (snapshot[(snapshot["window"] == w) & (snapshot["address"] == STRATWISE)].drop(columns=["address"]).round(6).to_dict(orient="records"))
                  for w in WINDOWS},
    "stratwise_age_days": int((LAST_MARK.floor("D") - daily[daily["address"] == STRATWISE]["date"].min()).days),
}
Path("_build/manifest_38.json").write_text(json.dumps(manifest, indent=1, default=str))
print("wrote _build/manifest_38.json")


wrote _build/manifest_38.json
